# 🏙️ Phân tích Chi tiết Top 10 Điểm Đen Kẹt Xe Nghiêm Trọng (Địa điểm duy nhất)
Notebook này thực hiện việc nhận diện các cụm kẹt xe (Black Spots) từ bộ dữ liệu GPS hành trình xe buýt, sau đó lọc ra **Top 10 địa danh duy nhất** có mức độ kẹt xe nghiêm trọng nhất để tránh việc lặp lại các điểm gần cùng một trạm xe buýt.

In [13]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MarkerCluster
from sklearn.metrics.pairwise import haversine_distances
import os

# Thiết lập đường dẫn tương đối (đảm bảo đang ở trong folder notebook)
DATA_DIR = "../data"
BLACKSPOT_FILE = os.path.join(DATA_DIR, "black_spot.parquet")
STATION_FILE = os.path.join(DATA_DIR, "2_silver", "bus_station_data.json")

print(f"📂 Kiểm tra file dữ liệu: {BLACKSPOT_FILE} - {'✅ Sẵn sàng' if os.path.exists(BLACKSPOT_FILE) else '❌ Không tìm thấy'}")

📂 Kiểm tra file dữ liệu: ../data\black_spot.parquet - ✅ Sẵn sàng


In [14]:
# 1. Tải dữ liệu
print("📥 Đang tải dữ liệu kẹt xe...")
df = pd.read_parquet(BLACKSPOT_FILE)
stations_df = pd.read_json(STATION_FILE)

# Đảm bảo định dạng tọa độ
df['x'] = df['x'].astype(float)
df['y'] = df['y'].astype(float)
stations_df['x'] = stations_df['x'].astype(float)
stations_df['y'] = stations_df['y'].astype(float)

print(f"📊 Tổng số tín hiệu kẹt xe: {len(df):,}")

📥 Đang tải dữ liệu kẹt xe...
📊 Tổng số tín hiệu kẹt xe: 933,436


In [15]:
# 2. Nhận diện các 'Điểm Đen' (Lọc địa điểm duy nhất)
print("🔍 Đang phân tích và lọc trùng địa điểm...")

# Bước 1: Gom nhóm theo Grid (11m x 11m)
df['lat_grid'] = df['y'].round(4)
df['lon_grid'] = df['x'].round(4)

counts = df.groupby(['lat_grid', 'lon_grid']).size().reset_index(name='severity')
all_candidates = counts.sort_values(by='severity', ascending=False).reset_index(drop=True)

def get_nearest_station(lat, lon, station_df):
    point = np.radians([[lat, lon]])
    station_coords = np.radians(station_df[['y', 'x']].values)
    dists = haversine_distances(point, station_coords)[0] * 6371000
    idx = np.argmin(dists)
    return station_df.iloc[idx]['Name']

# Bước 2: Gán tên trạm cho từng cụm và tính tọa độ thực tế
detailed_candidates = []
for idx, row in all_candidates.iterrows():
    mask = (df['lat_grid'] == row['lat_grid']) & (df['lon_grid'] == row['lon_grid'])
    actual_lat = df.loc[mask, 'y'].mean()
    actual_lon = df.loc[mask, 'x'].mean()
    loc_name = get_nearest_station(actual_lat, actual_lon, stations_df)
    detailed_candidates.append({
        'location_name': loc_name,
        'severity_score': row['severity'],
        'lat': actual_lat,
        'lon': actual_lon
    })

candidate_df = pd.DataFrame(detailed_candidates)

# Bước 3: LOẠI BỎ TRÙNG LẶP (Chỉ lấy điểm kẹt nặng nhất của mỗi trạm)
top_10 = candidate_df.sort_values(by='severity_score', ascending=False) \
                     .drop_duplicates(subset=['location_name']) \
                     .head(50) \
                     .reset_index(drop=True)

top_10['rank'] = top_10.index + 1

print("🏆 TOP 10 ĐIỂM ĐEN KẸT XE NGHIÊM TRỌNG (ĐỊA ĐIỂM DUY NHẤT)")
display(top_10[['rank', 'location_name', 'severity_score', 'lat', 'lon']])

🔍 Đang phân tích và lọc trùng địa điểm...
🏆 TOP 10 ĐIỂM ĐEN KẸT XE NGHIÊM TRỌNG (ĐỊA ĐIỂM DUY NHẤT)


,rank,location_name,severity_score,lat,lon
0,1,Trường Lê Quý Đôn,18111.0,10.769682,106.640194
1,2,Bãi xe buýt Phổ Quang,14607.0,10.805396,106.668022
2,3,Lâm Thành,8980.0,10.734212,106.614105
3,4,Ga metro Khu Công nghệ cao,8955.0,10.858314,106.786505
4,5,Bình Xuyên 2,8190.0,10.705600,106.715009
5,6,Cầu Bà Chiêm,7826.0,10.673495,106.731627
6,7,Ngã 3 Lam Sơn,6732.0,10.887798,106.575265
7,8,Trường Hoa Hồng Đỏ,4743.0,10.785027,106.793674
8,9,Nhà Tưởng niệm Liệt sỹ Nhà Bè,4708.0,10.602872,106.741588
9,10,Vành đai trong,4110.0,10.785048,106.792762


In [16]:
# 3. Trực quan hóa bản đồ không gian bằng Folium

# Khởi tạo bản đồ tại trung tâm thành phố
central_lat = top_10['lat'].mean() if not top_10.empty else 10.7769
central_lon = top_10['lon'].mean() if not top_10.empty else 106.7009
m = folium.Map(location=[central_lat, central_lon], zoom_start=13, tiles='CartoDB positron')

# a. Layer HeatMap cho tổng quan
heat_data = df[['y', 'x']].sample(min(50000, len(df))).values.tolist() 
HeatMap(heat_data, radius=8, blur=12, min_opacity=0.3, name="Mật độ kẹt xe").add_to(m)

# b. Marker cho Top 10 Điểm Đen Duy Nhất
for idx, row in top_10.iterrows():
    popup_html = f"""
    <div style=\"font-family: Arial, sans-serif; width: 220px;\">
        <h4 style=\"margin: 0; color: #d32f2f;\"># {row['rank']} - {row['location_name']}</h4>
        <hr style=\"margin: 5px 0;\">
        <p><b>🔥 Mức độ:</b> {row['severity_score']:,} tín hiệu</p>
        <p><b>🌐 Tọa độ:</b> {row['lat']:.5f}, {row['lon']:.5f}</p>
    </div>
    """
    
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"#{row['rank']} - {row['location_name']}",
        icon=folium.Icon(color='red', icon='fire', prefix='fa')
    ).add_to(m)

folium.LayerControl().add_to(m)

print("🗺️ Bản đồ đã sẵn sàng bên dưới:")
m

🗺️ Bản đồ đã sẵn sàng bên dưới:
